## 通过Agent在工具中使用长期记忆，基于InMemoryStore

In [1]:
from typing import NotRequired
from dotenv import load_dotenv
import os
from langchain_core.tools import tool
from langgraph.prebuilt import ToolRuntime
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True)

# 创建内存存储对象
store = InMemoryStore()
# 自定义一个继承于AgentState的类
class CustomState(AgentState):
    user_id : NotRequired[str]

# 1.实例化模型
model = ChatOpenAI(
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 保存用户信息到长期记忆中
@tool(parse_docstring=True)
def save_user_info(name:str,runtime:ToolRuntime):
    """
    将客户信息保存在长期记忆中

    Args:
        name : 用户名
        runtime : 工具的运行时

    Returns:
        str : 保存状态
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name": name}
    runtime.store.put(namespace, key, value)
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime:ToolRuntime):
    """
    从长期记忆中读取客户的信息

    Args:
        runtime : 工具的运行时

    Returns:
        str : 用户信息
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    item = runtime.store.get(namespace, key)
    return str(item.value) if item else "unknown"

## 创建agent
agent = create_agent(
    model=model,
    tools=[save_user_info, get_user_info],
    store=store,
    state_schema=CustomState,
    system_prompt="用户提及个人信息时，可以使用工具保存用户信息。如果用户询问个人信息时，可以尝试使用工具读取用户信息"
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)

resp1 = agent.invoke(
    {"messages":[HumanMessage("你你好，很高兴认识你，我是小花")],
     "user_id":"user_1"
     }
)

for msg in resp1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)

resp2 = agent.invoke(
    {"messages":[HumanMessage("我是谁？")],
     "user_id":"user_1"
     }
)

for msg in resp2["messages"]:
    msg.pretty_print()


============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你你好，很高兴认识你，我是小花
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_b9hcbwn3)
 Call ID: call_b9hcbwn3
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好，小花！很高兴认识你。有什么我可以帮你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁？
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_efrdjp8q)
 Call ID: call_efrdjp8q
  Args:
================================= Tool Message =================================
Name: get_user_info

{'name': '小花'